# Introdução ao Pandas: Qualidade do Ar em Bogotá (RMCAB)

Este notebook foi desenvolvido para a apresentação de mestrado sobre a biblioteca **Pandas**, utilizando o caso prático da Rede de Monitoramento da Qualidade do Ar de Bogotá (RMCAB).

**Objetivos:**
1. Operações básicas: Filtros, Agrupamentos, Estatísticas.
2. Ingestão de dados (CSV/Excel).
3. Preparação para Machine Learning (Fase B -> Fase C).
4. Visualização de resultados (Mapa de Calor).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium.plugins import HeatMap
import re
import os

# Funções Utilitárias do Projeto (Inlined)
def parse_dms(coor):
    if not isinstance(coor, str): return coor
    parts = re.split('[^\d\w]+', coor)
    degrees = float(parts[0])
    minutes = float(parts[1])
    seconds = float(parts[2]+'.'+parts[3])
    direction = parts[4]
    dec_coord = degrees + minutes / 60 + seconds / 3600
    if direction in ['S', 'W']:
        dec_coord *= -1
    return dec_coord

print("✅ Bibliotecas carregadas e funções utilitárias prontas!")

## 1. Carregamento de Dados (Fase B - Design)
Configuração flexível para rodar via Google Drive ou upload local.

In [ ]:
# --- CONFIGURAÇÃO DE AMBIENTE ---
USE_DRIVE = False  # Mude para True se você montou seu Drive no Colab

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    csv_path = '/content/drive/MyDrive/1-MestradoUFSC/IA_NA_BORDA/bogota_sensors_sample.csv'
else:
    csv_path = 'bogota_sensors_sample.csv'

# Verificação e Carga
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    # Convertendo coordenadas DMS para Decimal
    df['Lat_Dec'] = df['Latitude'].apply(parse_dms)
    df['Lon_Dec'] = df['Longitude'].apply(parse_dms)
    print(f"✅ Dados carregados com sucesso de: {csv_path}")
    display(df.head())
else:
    print(f"❗ ATENÇÃO: O arquivo '{csv_path}' não foi encontrado.")
    print("DICA: Faça o upload manual de 'bogota_sensors_sample.csv' ou mude USE_DRIVE para True.")

## 2. Operações Básicas (Pandas Fundamentals)
Explorando estatísticas descritivas e agrupamentos por estação.

In [ ]:
# 1. Estatísticas Descritivas Gerais
print("--- Estatísticas de Poluentes ---")
if 'df' in locals():
    print(df[['PM2.5', 'PM10', 'CO']].describe())

# 2. Agrupamento por Estação
print("\n--- Média de Poluição por Estação (GroupBy) ---")
if 'df' in locals():
    mean_pollution = df.groupby('Station')[['PM2.5', 'PM10']].mean().reset_index()
    display(mean_pollution)

## 3. Filtros e Alertas
Aplicação de filtros para detectar níveis críticos de poluição.

In [ ]:
if 'df' in locals():
    # Filtrando leituras de PM2.5 > 50 (Nível Crítico)
    alerta_pm25 = df[df['PM2.5'] > 50]
    print(f"Foram encontradas {len(alerta_pm25)} leituras em estado de Alerta.")
    display(alerta_pm25)

## 4. Avançando para IA (One-Hot & Temporal Features)
Como o Pandas prepara o terreno para a Fase C (Machine Learning).

In [ ]:
if 'df' in locals():
    # Criando variáveis temporais extras
    df['DateTime'] = pd.to_datetime(df['DateTime'])
    df['hora'] = df['DateTime'].dt.hour
    df['dia_semana'] = df['DateTime'].dt.dayofweek

    # One-Hot Encoding das Estações
    df_ml = pd.get_dummies(df, columns=['Station'], prefix='Estacao')

    print("Estrutura preparada para modelos preditivos:")
    display(df_ml.head())

## 5. Visualização Final (Fase C - Impacto)
Gerando o mapa de calor da poluição em Bogotá.

In [ ]:
if 'df' in locals():
    # Criando o mapa base centrado em Bogotá
    m = folium.Map(location=[4.6097, -74.0817], zoom_start=11, tiles='cartodbpositron')

    # Preparando dados para o HeatMap
    heat_data = [[row['Lat_Dec'], row['Lon_Dec'], row['PM2.5']] for index, row in df.iterrows()]

    # Adicionando a camada de calor
    HeatMap(heat_data, radius=25).add_to(m)
    display(m)